# Reference Model: MLP for Criterion Validity with GMDS

This notebook defines and validates the **standard criterion validity model** used throughout the thesis (Sections 6.1 and 6.2). Whenever a GMDS prediction experiment is reported, this is the model that is used.

## Model

A **shallow MLP regressor** trained to predict GMDS subscale scores from IRT–VAE latent ability scores, evaluated via **5-fold cross-validation**.

| Component | Choice | Rationale |
|---|---|---|
| Architecture | MLP, 1 hidden layer (32 units) | Low input dimensionality (~3 features); deeper nets overfit |
| Regularisation | L2 (`alpha=1e-4`) | Prevents overfitting on the limited GMDS sample (~560 children) |
| Evaluation | 5-fold CV, `shuffle=True, random_state=42` | Robust correlation estimate across random splits |
| Output | Pearson r, MSE, R² per GMDS subscale | Matches thesis Table 6.2 |

## Features used

- `tot` — MDAT total latent score (output of `VAE/mdat_scoring.ipynb`)
- `total_DEEP` — DEEP total latent score (output of `VAE/DEEP_scoring.ipynb`)
- `start_ability` — START latent score (output of `VAE/start_scoring.ipynb`)

Only three features are used to keep the mapping simple and avoid overfitting to the ~560 children who have both latent scores and GMDS ground truth.

## Inputs required

| File | Description |
|---|---|
| `mdat_Deep_abilities.csv` | MDAT latent scores — output of `VAE/mdat_scoring.ipynb` |
| `DEEP_Deep_abilities.csv` | DEEP latent scores — output of `VAE/DEEP_scoring.ipynb` |
| `start_Deep_abilities.csv` | START latent scores — output of `VAE/start_scoring.ipynb` |
| `gmds_scores.csv` | GMDS subscale raw scores: `child_ids`, `fol`, `langCom`, `eye_hand`, `soc_per`, `gross_mot` |

## Results (Table 6.2)

Mean Pearson r across 5 folds for each GMDS subscale:

| GMDS Subscale | Mean r | Std |
|---|---|---|
| Following instructions (`fol`) | 0.89 | 0.023 |
| Language & communication (`langCom`) | 0.84 | 0.027 |
| Eye–hand coordination (`eye_hand`) | 0.89 | 0.029 |
| Social & personal (`soc_per`) | 0.82 | 0.020 |
| Gross motor (`gross_mot`) | 0.87 | 0.027 |

In [15]:
import pandas as pd


target_cols = ["fol", "langCom", "eye_hand", "soc_per", "gross_mot"]
id_col = "child_ids"

gmds = pd.read_csv("gmds_scores.csv")
mdat = pd.read_csv("mdat_Deep_abilities.csv")
deep = pd.read_csv("DEEP_Deep_abilities.csv")
start = pd.read_csv("start_Deep_abilities.csv")

mdat = mdat[["child_ids", "Age", "tot"]]
mdat = mdat.drop_duplicates(subset=["child_ids"])

deep = deep[["child_ids", "Age", "total_DEEP"]]

start = start.drop_duplicates(subset=["child_ids"])

gmds.drop(columns=["Age"], inplace=True)
deep.drop(columns=["Age"], inplace=True)
start.drop(columns=["Age"], inplace=True)

df = pd.merge(start, deep, on="child_ids")
df = pd.merge(df, mdat, on="child_ids")
df = pd.merge(df, gmds, on="child_ids")

df.shape

(560, 10)

## Data Loading and Merging

Loads and merges the four input files. Note:
- MDAT: only the `tot` column (total latent score) is kept; duplicates on `child_ids` are dropped.
- DEEP: only `total_DEEP` is kept; `Age` is dropped before merging (GMDS Age is used).
- START: duplicates removed; `Age` dropped.
- GMDS: `Age` dropped to avoid conflicts.
- The final merged DataFrame has ~560 rows — children who appear in all four datasets.

In [16]:
# df.drop(columns=["fol", "langCom", "eye_hand", "gross_mot"], inplace=True, errors="ignore")
print(df.columns)

Index(['child_ids', 'start_ability', 'total_DEEP', 'Age', 'tot', 'fol',
       'langCom', 'eye_hand', 'soc_per', 'gross_mot'],
      dtype='object')


## Model: MLP Regressor (multi-output)

A single MLP is trained to predict all five GMDS subscales simultaneously. The `evaluate_model_per_target` function runs 5-fold cross-validation and reports per-fold and mean Pearson correlation, MSE, and R² for each GMDS subscale.

In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer

# Data
target_cols = ['fol', 'langCom', 'eye_hand', 'soc_per', 'gross_mot']
# target_cols = ["fm", "gm", "s", "l"]
# target_cols = ["soc_per"]
X = df.drop(columns=target_cols + ["child_ids", "Age"], errors="ignore")
y = df[target_cols]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
X.columns

Index(['start_ability', 'total_DEEP', 'tot'], dtype='object')

In [18]:
from sklearn.model_selection import KFold
from sklearn.base import clone
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, r2_score

def evaluate_model_per_target(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    fold_corrs = []
    fold_mses = []
    fold_r2s = []

    for train_idx, test_idx in kf.split(X):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model_clone = clone(model)
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)

        # Ensure 2D
        y_pred = np.atleast_2d(y_pred)
        if y_pred.shape[0] != len(X_test):
            y_pred = y_pred.T

        y_true = y_test.values
        y_true = np.atleast_2d(y_true)
        if y_true.shape[0] != len(X_test):
            y_true = y_true.T

        corrs = []
        mses = []
        r2s = []

        for i in range(y_true.shape[1]):
            corr = pd.Series(y_pred[:, i]).corr(
                pd.Series(y_true[:, i]), 
            )
            mse = mean_squared_error(y_true[:, i], y_pred[:, i])
            r2 = r2_score(y_true[:, i], y_pred[:, i])

            corrs.append(corr)
            mses.append(mse)
            r2s.append(r2)

        fold_corrs.append(corrs)
        fold_mses.append(mses)
        fold_r2s.append(r2s)

    fold_corrs = np.array(fold_corrs)
    fold_mses  = np.array(fold_mses)
    fold_r2s   = np.array(fold_r2s)

    results = pd.DataFrame({
        "Target": y.columns,
        "Mean Corr": np.round(fold_corrs.mean(axis=0), 2),
        "Std Corr":  np.round(fold_corrs.std(axis=0), 3),
        "Mean MSE":  fold_mses.mean(axis=0),
        "Mean R2":   fold_r2s.mean(axis=0)
    })

    return results


In [19]:
from sklearn.neural_network import MLPRegressor

mlp_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", MLPRegressor(
        hidden_layer_sizes=(32,),
        alpha=1e-4,
        max_iter=2000,
        random_state=42
    ))
])

mlp_results = evaluate_model_per_target(mlp_model, X, y)

print("Sklearn MLP")
display(mlp_results)


c:\Users\HP\miniconda3\envs\IRT\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


Sklearn MLP


c:\Users\HP\miniconda3\envs\IRT\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


,Target,Mean Corr,Std Corr,Mean MSE,Mean R2
0,fol,0.89,0.023,16.329790,0.778480
1,langCom,0.84,0.027,16.303894,0.709024
2,eye_hand,0.89,0.029,15.317464,0.784895
3,soc_per,0.82,0.020,20.536288,0.672557
4,gross_mot,0.87,0.027,13.749053,0.752664


In [20]:
import pandas as pd
import numpy as np
from scipy import stats

start = pd.read_csv("start_Deep_abilities.csv")  # or "start_Deep_abilities.csv" if that is your actual filename

x = start["start_ability"]
y = start["Age"]

valid = x.notna() & y.notna()
x = x[valid]
y = y[valid]

r, p = stats.pearsonr(x, y)
n = len(x)

# 95% CI using Fisher z
z = np.arctanh(r)
se = 1 / np.sqrt(n - 3)
z_crit = stats.norm.ppf(0.975)

ci_low, ci_high = np.tanh([z - z_crit * se, z + z_crit * se])

print(f"r = {r:.3f}")
print(f"95% CI = [{ci_low:.3f}, {ci_high:.3f}]")
print(f"p = {p:.4g}")
print(f"n = {n}")


r = 0.310
95% CI = [0.282, 0.338]
p = 1.156e-89
n = 3994
